In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
spark=SparkSession.builder.appName("Spark data").getOrCreate()

In [ ]:
# Define and use a complete schema for customers.csv.

customers=[(101,"Ramya","ramya@gmail.com","Hyderabad","2024-01-15",True),
            (102,"Anil",None,"Bangalore","2024-02-20",True),
            (103,"Sita","sita@gmail.com","Chennai","2024-03-10",False),
            (104,"Ravi","ravi@gmail.com","Mumbai","2024-04-05",True),
            (101,"Priya","priya@gmail.com","Delhi","2024-05-12",True),
            (105,"Kiran","kiran@gmail.com","Pune","2024-06-18",False),
            (104,"Meena","meena@gmail.com","Kolkata","2024-07-22",True),
            (108,"Arjun","arjun@gmail.com","Hyderabad","2024-08-09",True),
            (109,"Divya","divya@gmail.com","Bangalore","2024-09-14",True),
            (103,"Varun","varun@gmail.com",None,"2024-10-25",False)
]

schema=StructType([
    StructField("customer_id",IntegerType(),False),
    StructField("first_name",StringType(),False),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("registration_date",StringType(),False),
    StructField("is_active",BooleanType())
])

df_customers=spark.createDataFrame(customers,schema)
df_customers=df_customers.withColumn("registration_date",to_date(col("registration_date"),"yyyy-MM-dd"))
df_customers.printSchema()
df_customers.show()



root
 |-- customer_id: integer (nullable = false)
 |-- first_name: string (nullable = false)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = false)
 |-- is_active: boolean (nullable = true)

+-----------+----------+---------------+---------+-----------------+---------+
|customer_id|first_name|          email|     city|registration_date|is_active|
+-----------+----------+---------------+---------+-----------------+---------+
|        101|     Ramya|ramya@gmail.com|Hyderabad|       2024-01-15|     true|
|        102|      Anil|           NULL|Bangalore|       2024-02-20|     true|
|        103|      Sita| sita@gmail.com|  Chennai|       2024-03-10|    false|
|        104|      Ravi| ravi@gmail.com|   Mumbai|       2024-04-05|     true|
|        101|     Priya|priya@gmail.com|    Delhi|       2024-05-12|     true|
|        105|     Kiran|kiran@gmail.com|     Pune|       2024-06-18|    false|
|        104|     Meena|meena@gm

In [ ]:
from decimal import Decimal

In [ ]:
# Define a complete order_items schema with decimal prices and discounts.

order_items = [
    (1001, 5001, 201, 2, Decimal("499.99"), Decimal("10.50")),
    (1002, 5001, 202, 1, Decimal("1299.75"), Decimal("50.25")),
    (1003, 5002, 203, 5, Decimal("99.99"), Decimal("5.00")),
    (1004, 5002, 204, 3, Decimal("250.50"), Decimal("15.75")),
    (1005, 5003, 205, 1, Decimal("999.99"), Decimal("0.00")),
    (1006, 5003, 201, 4, Decimal("499.99"), Decimal("25.50")),
    (1007, 5004, 206, 2, Decimal("750.25"), Decimal("20.00")),
    (1008, 5005, 207, 1, Decimal("1599.90"), Decimal("100.50")),
    (1009, 5005, 208, 6, Decimal("75.25"), Decimal("5.25")),
    (1010, 5006, 209, 2, Decimal("349.99"), None),
    (1011, 5007, 210, 3, Decimal("450.75"), Decimal("30.25")),
    (1012, 5008, 203, 2, Decimal("99.99"), Decimal("10.00")),
    (1013, 5009, 211, 1, Decimal("2499.99"), Decimal("150.75")),
    (1014, 5010, 212, 5, Decimal("125.50"), Decimal("12.50")),
    (1015, 5010, 201, 2, Decimal("499.99"), Decimal("50.00"))
]
schema=StructType([
    StructField("order_item_id",IntegerType(),False),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2),True)
])
df_order_items=spark.createDataFrame(order_items,schema)
df_order_items.printSchema()
df_order_items.show()

root
 |-- order_item_id: integer (nullable = false)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount: decimal(10,2) (nullable = true)

+-------------+--------+----------+--------+----------+--------+
|order_item_id|order_id|product_id|quantity|unit_price|discount|
+-------------+--------+----------+--------+----------+--------+
|         1001|    5001|       201|       2|    499.99|   10.50|
|         1002|    5001|       202|       1|   1299.75|   50.25|
|         1003|    5002|       203|       5|     99.99|    5.00|
|         1004|    5002|       204|       3|    250.50|   15.75|
|         1005|    5003|       205|       1|    999.99|    0.00|
|         1006|    5003|       201|       4|    499.99|   25.50|
|         1007|    5004|       206|       2|    750.25|   20.00|
|         1008|    5005|       207|       1|   1599.90|  100.50|
|         1

In [ ]:
# Read orders with order_ts parsed as a timestamp in UTC.

orders = [
    (5001, 101, "2026-10-05 10:30:00", 2299.49),
    (5002, 102, "2026-10-05 14:45:30", 799.25),
    (5003, 103, "2026-10-06 09:15:10", 2974.48),
    (5004, 104, "2026-10-06 11:20:45", 1480.50),
    (5005, 101, "2026-10-07 16:10:00", None),
    (5006, 105, "2026-10-07 18:35:20", 699.98),
    (5007, 104, "2026-10-08 08:45:15", 1322.75),
    (5008, 106, "2026-10-08 13:25:40", 199.98),
    (5009, 107, "2026-10-09 15:50:05", 2349.24),
    (5010, 108, "2026-10-09 20:15:30", 875.50)
]
schema=StructType([
    StructField("order_id",IntegerType(),False),
    StructField("customer_id",IntegerType(),False),
    StructField("order_ts",StringType(),False),
    StructField("total_amount",DoubleType(),True)

])
df_orders=spark.createDataFrame(orders,schema)
df_orders=df_orders.withColumn("order_ts",to_timestamp(col("order_ts"),"yyyy-MM-dd HH:mm:ss"))
df_orders.printSchema()
df_orders.show()

root
 |-- order_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- order_ts: timestamp (nullable = false)
 |-- total_amount: double (nullable = true)

+--------+-----------+-------------------+------------+
|order_id|customer_id|           order_ts|total_amount|
+--------+-----------+-------------------+------------+
|    5001|        101|2026-10-05 10:30:00|     2299.49|
|    5002|        102|2026-10-05 14:45:30|      799.25|
|    5003|        103|2026-10-06 09:15:10|     2974.48|
|    5004|        104|2026-10-06 11:20:45|      1480.5|
|    5005|        101|2026-10-07 16:10:00|        NULL|
|    5006|        105|2026-10-07 18:35:20|      699.98|
|    5007|        104|2026-10-08 08:45:15|     1322.75|
|    5008|        106|2026-10-08 13:25:40|      199.98|
|    5009|        107|2026-10-09 15:50:05|     2349.24|
|    5010|        108|2026-10-09 20:15:30|       875.5|
+--------+-----------+-------------------+------------+



In [ ]:
df_customers.createOrReplaceTempView("customers")
df_order_items.createOrReplaceTempView("order_items")
df_orders.createOrReplaceTempView("orders")
display(df_customers)
display(df_order_items)
display(df_orders)

DataFrame[customer_id: int, first_name: string, email: string, city: string, registration_date: date, is_active: boolean]

DataFrame[order_item_id: int, order_id: int, product_id: int, quantity: int, unit_price: decimal(10,2), discount: decimal(10,2)]

DataFrame[order_id: int, customer_id: int, order_ts: timestamp, total_amount: double]

In [ ]:
# Compare inferred and explicit schemas for the same CSV file.
df_customers.write.csv("data.csv/df_customers.csv",header=True,mode="overwrite")
df_customers_infer=spark.read.csv("data.csv/df_customers.csv",header=True,inferSchema=True)
df_customers_infer.show()
df_customers_infer.printSchema()
df_customers.printSchema()


+-----------+----------+---------------+---------+-----------------+---------+
|customer_id|first_name|          email|     city|registration_date|is_active|
+-----------+----------+---------------+---------+-----------------+---------+
|        105|     Kiran|kiran@gmail.com|     Pune|       2024-06-18|    false|
|        104|     Meena|meena@gmail.com|  Kolkata|       2024-07-22|     true|
|        108|     Arjun|arjun@gmail.com|Hyderabad|       2024-08-09|     true|
|        109|     Divya|divya@gmail.com|Bangalore|       2024-09-14|     true|
|        103|     Varun|varun@gmail.com|     NULL|       2024-10-25|    false|
|        101|     Ramya|ramya@gmail.com|Hyderabad|       2024-01-15|     true|
|        102|      Anil|           NULL|Bangalore|       2024-02-20|     true|
|        103|      Sita| sita@gmail.com|  Chennai|       2024-03-10|    false|
|        104|      Ravi| ravi@gmail.com|   Mumbai|       2024-04-05|     true|
|        101|     Priya|priya@gmail.com|    Delhi|  

In [ ]:
# Read a CSV fixture containing a quoted comma in a customer name.
df_customers=spark.read.option("header","true").option("quote",'"').csv("data.csv/df_customers.csv")
df_customers.show()

+-----------+----------+---------------+---------+-----------------+---------+
|customer_id|first_name|          email|     city|registration_date|is_active|
+-----------+----------+---------------+---------+-----------------+---------+
|        105|     Kiran|kiran@gmail.com|     Pune|       2024-06-18|    false|
|        104|     Meena|meena@gmail.com|  Kolkata|       2024-07-22|     true|
|        108|     Arjun|arjun@gmail.com|Hyderabad|       2024-08-09|     true|
|        109|     Divya|divya@gmail.com|Bangalore|       2024-09-14|     true|
|        103|     Varun|varun@gmail.com|     NULL|       2024-10-25|    false|
|        101|     Ramya|ramya@gmail.com|Hyderabad|       2024-01-15|     true|
|        102|      Anil|           NULL|Bangalore|       2024-02-20|     true|
|        103|      Sita| sita@gmail.com|  Chennai|       2024-03-10|    false|
|        104|      Ravi| ravi@gmail.com|   Mumbai|       2024-04-05|     true|
|        101|     Priya|priya@gmail.com|    Delhi|  

In [ ]:
# import shutil
# shutil.rmtree("customers_empty.csv")

In [ ]:
# Read a header-only CSV with an explicit schema and verify zero rows.
customers_empty=[]
schema=StructType([
    StructField("customer_id",IntegerType(),False),
    StructField("first_name",StringType(),False),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("registration_date",StringType(),False),
    StructField("is_active",BooleanType())
])
df_customers_empty=spark.createDataFrame(customers_empty,schema)
df_customers_empty.write.csv("df_customers_empty.csv",header=True,mode="overwrite")
df_customers_empty=spark.read.option("header","true").schema(schema).csv("df_customers_empty.csv")
df_customers_empty.show()
print("Number of rows:",df_customers_empty.count())

+-----------+----------+-----+----+-----------------+---------+
|customer_id|first_name|email|city|registration_date|is_active|
+-----------+----------+-----+----+-----------------+---------+
+-----------+----------+-----+----+-----------------+---------+

Number of rows: 0


In [ ]:
# Create a malformed numeric fixture and inspect PERMISSIVE output.
order_malform=[
    (5001,101,2,2299.49),
(5002,102,1,799.25),
(5003,103,5,2974.48),
(5004,104,3,1480.50),
(5005,105,"abc",699.98),
(5006,106,4,1250.75),
(5007,107,"xyz",899.99),
(5008,108,2,450.50),
(5009,109,10,1500.25),
(5010,110,6,999.99)
]

schema_string = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("quantity", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])
df_orders_temp=spark.createDataFrame(order_malform,schema_string)
# df_order_malform=spark.createDataFrame(order_malform,schema)
schema=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("total_amount",DoubleType(),True)
    ])

df_orders_temp.write.csv("orders_malformed.csv",header=True,mode="overwrite")
df_order_malform=spark.read.option("header","true").schema(schema).option("mode","permissive").csv("orders_malformed.csv")
df_order_malform.show()

+--------+-----------+--------+------------+
|order_id|customer_id|quantity|total_amount|
+--------+-----------+--------+------------+
|    5001|        101|       2|     2299.49|
|    5002|        102|       1|      799.25|
|    5003|        103|       5|     2974.48|
|    5004|        104|       3|      1480.5|
|    5005|        105|    NULL|      699.98|
|    5006|        106|       4|     1250.75|
|    5007|        107|    NULL|      899.99|
|    5008|        108|       2|       450.5|
|    5009|        109|      10|     1500.25|
|    5010|        110|       6|      999.99|
+--------+-----------+--------+------------+



In [ ]:
# # Compare DROPMALFORMED and FAILFAST on that fixture using an action.
# df_order_malform.filter(col("quantity").isNotNull()).show()
order_malform=[
    (5001,101,2,2299.49),
(5002,102,1,799.25),
(5003,103,5,2974.48),
(5004,104,3,1480.50),
(5005,105,"abc",699.98),
(5006,106,4,1250.75),
(5007,107,"xyz",899.99),
(5008,108,2,450.50),
(5009,109,10,1500.25),
(5010,110,6,999.99)
]

schema_string = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("quantity", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])
df_orders_temp=spark.createDataFrame(order_malform,schema_string)
# df_order_malform=spark.createDataFrame(order_malform,schema)
schema=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("total_amount",DoubleType(),True)
    ])

df_orders_temp.write.csv("orders_malformed.csv",header=True,mode="overwrite")
df_order_malform=spark.read.option("header","true").schema(schema).option("mode","permissive").option("mode","Dropmalformed").csv("orders_malformed.csv")
# df_fail=spark.read.option("header","true").option("mode","FAILFAST").option("mode","Dropmalformed").schema(schema).csv("orders_malformed.csv")
df_order_malform.show()

# Fail Fast
df_fail=spark.read.option("header","true").option("mode","FAILFAST").schema(schema).csv("orders_malformed.csv")
df_fail.show()

+--------+-----------+--------+------------+
|order_id|customer_id|quantity|total_amount|
+--------+-----------+--------+------------+
|    5001|        101|       2|     2299.49|
|    5002|        102|       1|      799.25|
|    5003|        103|       5|     2974.48|
|    5004|        104|       3|      1480.5|
|    5006|        106|       4|     1250.75|
|    5008|        108|       2|       450.5|
|    5009|        109|      10|     1500.25|
|    5010|        110|       6|      999.99|
+--------+-----------+--------+------------+



Py4JJavaError: An error occurred while calling o399.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///content/orders_malformed.csv/part-00000-8879e44e-8fe4-4260-9d1c-7445b24e68b3-c000.csv.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:856)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:583)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeSt

In [ ]:
df_empty_stamp=df_customers_empty.withColumn("input_file_name",input_file_name())\
                  .withColumn("ingestion_timestamp",current_timestamp())
df_empty_stamp.show()

+-----------+----------+-----+----+-----------------+---------+---------------+-------------------+
|customer_id|first_name|email|city|registration_date|is_active|input_file_name|ingestion_timestamp|
+-----------+----------+-----+----+-----------------+---------+---------------+-------------------+
+-----------+----------+-----+----+-----------------+---------+---------------+-------------------+



In [ ]:
# Add source filename and ingestion timestamp audit columns.
df_timestamp=df_order_malform.withColumn("input_file_name",input_file_name())\
                              .withColumn("ingestion_timestamp",current_timestamp())
df_timestamp.show()

+--------+-----------+--------+------------+--------------------+--------------------+
|order_id|customer_id|quantity|total_amount|     input_file_name| ingestion_timestamp|
+--------+-----------+--------+------------+--------------------+--------------------+
|    5001|        101|       2|     2299.49|file:///content/o...|2026-10-06 05:21:...|
|    5002|        102|       1|      799.25|file:///content/o...|2026-10-06 05:21:...|
|    5003|        103|       5|     2974.48|file:///content/o...|2026-10-06 05:21:...|
|    5004|        104|       3|      1480.5|file:///content/o...|2026-10-06 05:21:...|
|    5006|        106|       4|     1250.75|file:///content/o...|2026-10-06 05:21:...|
|    5008|        108|       2|       450.5|file:///content/o...|2026-10-06 05:21:...|
|    5009|        109|      10|     1500.25|file:///content/o...|2026-10-06 05:21:...|
|    5010|        110|       6|      999.99|file:///content/o...|2026-10-06 05:21:...|
+--------+-----------+--------+------------

In [ ]:
# from IPython.terminal.interactiveshell import Integer
# Read one event archive file using the supplied nested event_schema.
event={"event_id":"E002","event_type":"login","user":{"user_id":102,"name":"Anil"}}
# df_events.show()
event_schema=StructType([
    StructField("event_id",StringType(),True),
    StructField("event_type",StringType(),True),
    StructField("user",StructType([
         StructField("user_id",IntegerType(),True),
         StructField("name",StringType(),True)
         ]))
])
df_event=spark.createDataFrame([event],event_schema)
# df_event.write.json("df_event.json")
df_event=spark.read.schema(event_schema).json("df_event.json")
df_event.show()
df_event.printSchema()

#Advanced challenges (21-30)


In [31]:
# Preserve malformed CSV evidence using a configured corrupt-record column.
order_malform=[
    (5001,101,2,2299.49),
(5002,102,1,799.25),
(5003,103,5,2974.48),
(5004,104,3,1480.50),
(5005,105,"abc",699.98),
(5006,106,4,1250.75),
(5007,107,"xyz",899.99),
(5008,108,2,450.50),
(5009,109,10,1500.25),
(5010,110,6,999.99)
]

schema_corrupt=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("total_amount",DoubleType(),True),
    StructField("_corrupt_record",StringType(),True)
    ])

df_orders_temp.write.csv("orders_malformed.csv",header=True,mode="overwrite")
# df_order_malform=spark.read.option("header","true").schema(schema).option("mode","permissive").option("mode","Dropmalformed").csv("orders_malformed.csv")
# df_order_malform.show()
df_corrupt=spark.read.option("header","true").option("mode","permissive").option("ColumnNameOfCorruptRecord","_corrupt_record").schema(schema_corrupt).csv("orders_malformed.csv")
df_corrupt.show()


+--------+-----------+--------+------------+-------------------+
|order_id|customer_id|quantity|total_amount|    _corrupt_record|
+--------+-----------+--------+------------+-------------------+
|    5001|        101|       2|     2299.49|               NULL|
|    5002|        102|       1|      799.25|               NULL|
|    5003|        103|       5|     2974.48|               NULL|
|    5004|        104|       3|      1480.5|               NULL|
|    5005|        105|    NULL|      699.98|5005,105,abc,699.98|
|    5006|        106|       4|     1250.75|               NULL|
|    5007|        107|    NULL|      899.99|5007,107,xyz,899.99|
|    5008|        108|       2|       450.5|               NULL|
|    5009|        109|      10|     1500.25|               NULL|
|    5010|        110|       6|      999.99|               NULL|
+--------+-----------+--------+------------+-------------------+



#Explain why fewer or extra CSV tokens are not always classified as corrupt records.

Fewer or extra CSV tokens are not always considered corrupt records because Spark's CSV parser can sometimes handle missing or extra fields according to its parsing rules. A corrupt record generally refers to a row that cannot be properly parsed according to the CSV format, such as malformed quoting or other structural parsing errors.


#Demonstrate how column pruning can affect CSV corruption detection.

Column Pruning means spark reads only the columns required for the query . if a malformed value exists in a column that is pruned / nor selected , Spark may not inspect that value, so the corruption may not be detected . if that column is selected, spark has to parse it and the malformed value can be detected

#Compare missing columns, reordered columns, and incompatible types during ingestion.

Missing columns mean the source doesn't provide a field expected by the schema so the spark can fill it with NULL  

Reordered columns are dangerous in CSV because CSV parsing is position based so values can map to the wrong schema field if the order differs.

Incompatible types occur when a value cannot be converted to the expected data type such as abc for integerType , and the result depends on the parsing mode

#Design a contract that distinguishes parsing failures from business-rule failures.


I would separate ingestion validation into two layers.

The parsing layer validates whether the source data can be correctly parsed according to the schema.

The business-Validation layer checks whether successfully parsed data satisfies business rules .Parsing failure would be classified as parse_error, while rule violations would classified as business_rule_error.This separation makes troubleshooting,monitoring and data-quality reporting  easier

#Explain why nullable=False is not a complete data-quality enforcement strategy.

nullable = false only describes whether the null values are allowed in the schema . it does not validate the correctness,range,format,uniqueness or business meaning of the date. Therefore , it should be combined with parsing validation and explicit data quality or business rule checks.


In [ ]:
# Preserve an identifier with leading zeros through read and write.
customer_zeros=[
    ("00101","Ramya","ramya@gmail.com"),
    ("00102","Naidu","naidu123@gmail.com"),
    ("00103","Rupesh","rupesh@outlook.com")
]
schema=StructType([
    StructField("customer_id",StringType(),True),
    StructField("name",StringType(),True),
    StructField("email_id",StringType(),True)
    ])
df_customer_zero=spark.createDataFrame(customer_zeros,schema)
df_customer_zero.write.csv("customer_zeros.csv",header=True,mode="overwrite")
df_customer_zero=spark.read.csv("customer_zeros.csv",header=True)
df_customer_zero.show()

In [ ]:
# Create a decimal-overflow fixture and record parsing behavior under your configuration.
data = [
    (Decimal("101.25"),),
    (Decimal("9999999999999999999999.99"),),  # overflow value
    (Decimal("50.75"),)
]
schema=StructType([
    StructField("amount",DecimalType(10,2),True)
])
df_data=spark.createDataFrame(data,schema)
df_data.show()

In [37]:
# Design ingestion that retains raw text for rows that cannot be typed safely.
customer_data=[
    ("C101",10),
    ("C102","abc"),
    ("C103",20)
]
schema=StructType([
    StructField("customer_id",StringType(),True),
    StructField("quantity_raw",StringType(),False)
])
df_customer_data=spark.createDataFrame(customer_data,schema)
df_customer_data.show()
df_customer_data.printSchema()
df_customer_data_convert=df_customer_data.withColumn("quantity",col("quantity_raw").try_cast("int"))
df_customer_data_convert.printSchema()
df_customer_data_convert.show()


+-----------+------------+
|customer_id|quantity_raw|
+-----------+------------+
|       C101|          10|
|       C102|         abc|
|       C103|          20|
+-----------+------------+

root
 |-- customer_id: string (nullable = true)
 |-- quantity_raw: string (nullable = false)

root
 |-- customer_id: string (nullable = true)
 |-- quantity_raw: string (nullable = false)
 |-- quantity: integer (nullable = true)

+-----------+------------+--------+
|customer_id|quantity_raw|quantity|
+-----------+------------+--------+
|       C101|          10|      10|
|       C102|         abc|    NULL|
|       C103|          20|      20|
+-----------+------------+--------+



In [ ]:
# Create a malformed numeric fixture and inspect PERMISSIVE output.
order_malform=[
    (5001,101,2,2299.49),
(5002,102,1,799.25),
(5003,103,5,2974.48),
(5004,104,3,1480.50),
(5005,105,"abc",699.98),
(5006,106,4,1250.75),
(5007,107,"xyz",899.99),
(5008,108,2,450.50),
(5009,109,10,1500.25),
(5010,110,6,999.99)
]

schema_string = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("quantity", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])
df_orders_temp=spark.createDataFrame(order_malform,schema_string)
# df_order_malform=spark.createDataFrame(order_malform,schema)
schema=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("total_amount",DoubleType(),True)
    ])

df_orders_temp.write.csv("orders_malformed.csv",header=True,mode="overwrite")
df_order_malform=spark.read.option("header","true").schema(schema).option("mode","permissive").csv("orders_malformed.csv")
df_order_malform.show()

In [81]:
# Write a reusable reader with schema, format, audit columns, and parse-mode arguments.
invalid_data=[
    ("C101","Alice",10,Decimal("2500.50"),"2026-10-01"),
    ("C102","Bob",5,Decimal("1200.00"),"2026-10-02"),
    ("C103","Charlie","abc",Decimal("1800.75"),"2026-10-03"),
    ("C104","David",8,"invalid","2026-10-04"),
    ("C105","Eva",12,Decimal("3200.00"),"invalid-date"),
    ("C106","Frank",7,Decimal("2100.25"),"2026-10-06")
]
schema=StructType([
    StructField("customer_id",StringType(),False),
    StructField("first_name",StringType(),True),
    StructField("quantity_raw",StringType(),True),
    StructField("amount_raw",StringType(),True),
    StructField("order_date_raw",StringType(),True)
    ])
df_invalid_data=spark.createDataFrame(invalid_data,schema)
df_invalid_data.show()
df_invalid_data.printSchema()

# ---------------------------------------------------
# 2. Write the RAW data to CSV
# ---------------------------------------------------
df_invalid_data.write.csv("data.raw/df_invalid_data",header=True,mode="overwrite")

# ---------------------------------------------------
# 4. Read using PERMISSIVE mode
# ---------------------------------------------------


df_permissive=spark.read.option("header","true").option("mode","permissive").csv("data.raw/df_invalid_data")
print("Permissive data")
df_permissive.show()
df_permissive.printSchema()


# ---------------------------------------------------
# 5. Safely convert raw columns to required types
# ---------------------------------------------------


df_invalid_data_schema=df_invalid_data.withColumn("quantity",col("quantity_raw").try_cast("int"))\
                                      .withColumn("amount",col("amount_raw").try_cast("Decimal(10,2)"))\
                                      .withColumn("order_date",col("order_date_raw").try_cast("date"))\

df_invalid_data_schema.printSchema()
df_invalid_data_schema.show()
df_invalid_data_schema.printSchema()


# ---------------------------------------------------
# 6. Add audit columns
# ---------------------------------------------------

df_invalid_data_schema.write.csv("data.invalid/df_invalid_data",header=True,mode="overwrite")
df_invalid_data_schema=spark.read.csv("data.invalid/df_invalid_data",header=True)
df_invalid_data_schema=df_invalid_data_schema.withColumn("input_file_name",input_file_name())\
                                             .withColumn("ingestion_timestamp",current_timestamp())
df_invalid_data_schema.show(truncate=False)

# ---------------------------------------------------
# 7. Read using DROPMALFORMED
# ---------------------------------------------------

#Dropmalformed
df_dropmalformed=spark.read.option("header","true").option("mode","permissive").option("mode","Dropmalformed").csv("data.raw/df_invalid_data")

# df_invalid_date_schema=spark.read.option("header","true").option("mode","Dropmalformed").csv("data.invalid/df_invalid_data")
df_dropmalformed.show()





+-----------+----------+------------+----------+--------------+
|customer_id|first_name|quantity_raw|amount_raw|order_date_raw|
+-----------+----------+------------+----------+--------------+
|       C101|     Alice|          10|   2500.50|    2026-10-01|
|       C102|       Bob|           5|   1200.00|    2026-10-02|
|       C103|   Charlie|         abc|   1800.75|    2026-10-03|
|       C104|     David|           8|   invalid|    2026-10-04|
|       C105|       Eva|          12|   3200.00|  invalid-date|
|       C106|     Frank|           7|   2100.25|    2026-10-06|
+-----------+----------+------------+----------+--------------+

root
 |-- customer_id: string (nullable = false)
 |-- first_name: string (nullable = true)
 |-- quantity_raw: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- order_date_raw: string (nullable = true)

Permissive data
+-----------+----------+------------+----------+--------------+
|customer_id|first_name|quantity_raw|amount_raw|order_d